# Whisper-small fine-tune (Colab wrapper)

This notebook **only wraps** the `pasak` package. There is no second training loop here.

**Runtime → Change runtime type → GPU** (T4 is enough).

**Do not zip FLEURS from the PC.** `prepare_asr --fleurs` pulls `google/fleurs` `id_id` on this VM.

Train = FLEURS train (+ optional Common Voice) **plus** expense golden 80% mix-in. Golden-only is refused (`n < 50` ASR train → exit `5`). FLEURS test is never trained.

Upload two zips to `/content` (Files pane):
1. `backend-src.zip` — `pyproject.toml` + `src/` (+ `notebooks/` optional). Use the Python zipper on Windows (below), not `Compress-Archive` (backslash paths).
2. `golden-v0.zip` — from `python -m pasak.pack_golden` on the PC (`backend/results/golden-v0.zip`). Contains `examples.jsonl` + `audio/*.wav`.

Checkpoint leaves the VM via **R2** under `maintainer/whisper-small-ft-golden/` (does not overwrite the FLEURS-only prefix). Colab Secrets: `R2_ACCOUNT_ID`, `R2_BUCKET`, `R2_ACCESS_KEY_ID`, `R2_SECRET_ACCESS_KEY`, Notebook access ON.

In [ ]:
import zipfile
from pathlib import Path


def unzip_win(zip_path: str, dest: str) -> None:
    root = Path(dest)
    root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        for info in z.infolist():
            name = info.filename.replace("\\", "/")
            if name.endswith("/"):
                continue
            out = root / name
            out.parent.mkdir(parents=True, exist_ok=True)
            out.write_bytes(z.read(info.filename))
    print("extracted", zip_path, "->", dest, list(root.iterdir())[:8])


unzip_win("/content/backend-src.zip", "/content/backend")
assert (Path("/content/backend") / "pyproject.toml").is_file(), "pyproject.toml missing; zip src+toml into backend-src.zip"
assert (Path("/content/backend") / "src" / "pasak").is_dir()

In [ ]:
%pip install -e "/content/backend[ml]"

If the install says restart the session: **Runtime → Restart session**, re-run the pip cell once, then continue. Do not skip GPU.

In [ ]:
# Optional: if google/fleurs is gated
# %pip install huggingface_hub
# from huggingface_hub import login
# login()

!python -m pasak.prepare_asr --fleurs
# Optional CV dump uploaded to /content/cv: add --cv-dir /content/cv

In [ ]:
unzip_win("/content/golden-v0.zip", "/content/golden-v0")
assert Path("/content/golden-v0/examples.jsonl").is_file()
assert any(Path("/content/golden-v0/audio").glob("*.wav"))

In [ ]:
!python -m pasak.finetune_whisper \
  --asr /content/backend/data/asr/manifest.jsonl \
  --golden /content/golden-v0/examples.jsonl \
  --out /content/whisper-small-ft-golden \
  --epochs 3

## Upload checkpoint to R2

Left sidebar **key** icon. Secrets: `R2_ACCOUNT_ID`, `R2_BUCKET`, `R2_ACCESS_KEY_ID`, `R2_SECRET_ACCESS_KEY`. Notebook access ON.

Prefix `maintainer/whisper-small-ft-golden/` — FLEURS-only weights stay at `maintainer/whisper-small-ft/`.

In [ ]:
%pip install -q boto3

from pathlib import Path

import boto3
from botocore.config import Config
from google.colab import userdata

account = userdata.get("R2_ACCOUNT_ID")
bucket = userdata.get("R2_BUCKET")
prefix = "maintainer/whisper-small-ft-golden/"
root = Path("/content/whisper-small-ft-golden")

s3 = boto3.client(
    "s3",
    endpoint_url=f"https://{account}.r2.cloudflarestorage.com",
    aws_access_key_id=userdata.get("R2_ACCESS_KEY_ID"),
    aws_secret_access_key=userdata.get("R2_SECRET_ACCESS_KEY"),
    region_name="auto",
    config=Config(s3={"addressing_style": "path"}),
)

for path in root.rglob("*"):
    if not path.is_file():
        continue
    if any(part.startswith("checkpoint-") for part in path.relative_to(root).parts):
        continue
    key = prefix + path.relative_to(root).as_posix()
    print("put", key, path.stat().st_size)
    s3.upload_file(str(path), bucket, key)
print("done")